# 第七轮：频带、时长和顺序的关联记忆
本轮先验证输入后的内部状态能否保留两段频带各自的时长；不训练自主复述。模型逐帧接收输入，诊断读出器只访问固定容量的最终关联状态。训练含有顺序位置轨迹的模型和不推进位置的对照模型。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Parrot')
out = root / 'trial7_results'
out.mkdir(parents=True, exist_ok=True)


## 快速检查


In [ ]:
!python trial7.py --smoke --batch 4 --out /content/Parrot/trial7_smoke


## 正式训练与评估
评估包含等待 0、16、64 帧，重复频带和交换两段时长的反事实样本。重点比较 `ordered` 与 `pooled` 的联合正确率和反事实成对正确率。


In [ ]:
!python trial7.py --steps 2000 --batch 64 --out {out}


In [ ]:
import json
report_path = out / 'metrics.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
report['git_commit'] = commit
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))


## 回传
将 ZIP 放入项目 `result/` 目录。


In [ ]:
import shutil
archive = shutil.make_archive(str(root / 'trial7_results'), 'zip', root_dir=str(root), base_dir='trial7_results')
print(archive)
